# Deep Graph Infomax (Transductive Cora)

Unsupervised Representation on Cora (Planetoid): Unsupervised node embeddings by contrasting local vs corrupted global graph representations. This notebook implements the approach with `DeepGraphInfomax` inside a `GCNEncoder` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DeepGraphInfomax` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models
from k3_node.datasets import Planetoid

title = "Deep Graph Infomax (Transductive Cora) with GCNConv"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]

# 2. Encoder & DGI Model
class GCNEncoder(keras.Model):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv = k3_layers.GCNConv(in_channels, hidden_channels)

    def call(self, x, edge_index):
        return ops.relu(self.conv(x, edge_index))

def summary_fn(z, *args, **kwargs):
    return ops.sigmoid(ops.mean(z, axis=0))

def corruption_fn(x, edge_index):
    indices = keras.random.shuffle(ops.arange(ops.shape(x)[0]))
    return ops.take(x, indices, axis=0), edge_index

encoder = GCNEncoder(dataset.num_features, 64)
model = k3_models.DeepGraphInfomax(
    hidden_channels=64,
    encoder=encoder,
    summary=summary_fn,
    corruption=corruption_fn,
)

# 3. Eager Verification
pos_z, neg_z, summary = model(data.x, data.edge_index)
print(f"Transductive DGI positive latent representations: {pos_z.shape}")

print("\n✓ K3-Node DeepGraphInfomax (Transductive) execution completed successfully!")